# HDS → TouchSim

HDS pattern (binary 5×3 tensor) → TouchSim → spike counts → similarity between digits.

- TouchSim is not on PyPI; it is installed from GitHub `hsaal/touchsim`. Never overwrite `ts.hand_surface.pxl_per_mm` (≈ 2.18 px/mm, TouchSim hand ≈ 209 mm).
- Only the hand is rescaled: `HAND_SCALE = 189 / TouchSim hand length ≈ 0.904`. The HDS keeps its real size (50×80 mm, pin Ø10, pitch 15, depth 1 mm), so 1 real mm = `ts_px_per_mm / HAND_SCALE` ≈ 2.41 px.
- For TouchSim, pin positions and pin radius are divided by `HAND_SCALE`. Depth stays 1 mm.
- **Row-by-row presentation:** row 0 (top) first, then the next rows. Each row: 0.2 s ramp down, 0.6 s hold, 0.2 s ramp up (1.0 s), all active pins of that row together; the next row starts when the previous one is back up. A 5-row pattern lasts 5 s. The stimulus is built as one `ts.Stimulus` whose per-pin trace is zero outside its row's window.
- Afferents are subsampled to SA1 : RA : PC = 2 : 4 : 1 (Saal et al., 2017); `make_population(class_ratio=None)` disables this.
- Coordinates: *pixel* (hand image) and *hand* (mm, used by Stimulus/Afferent), converted by `hand_surface.pixel2hand / hand2pixel`.

In [ ]:
!pip install -q git+https://github.com/hsaal/touchsim.git
import os
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon, Rectangle
import touchsim as ts

## 1. HDS: specification, digit patterns, geometry

In [ ]:
ROWS, COLS = 5, 3


@dataclass(frozen=True)
class HDSSpec:
    pin_diameter_mm: float = 10.0
    pitch_mm: float = 15.0
    depth_mm: float = 1.0
    ramp_s: float = 0.2
    hold_s: float = 0.6
    width_mm: float = 50.0
    height_mm: float = 80.0
    rows: int = ROWS
    cols: int = COLS

    @property
    def row_duration_s(self) -> float:
        """One row: ramp down + hold + ramp up."""
        return 2 * self.ramp_s + self.hold_s

    @property
    def duration_s(self) -> float:
        """Whole pattern: rows are presented one after another."""
        return self.rows * self.row_duration_s

    def pin_offsets_mm(self) -> np.ndarray:
        """(rows, cols, 2) pin centers in mm relative to the display center (x right, y up)."""
        xs = (np.arange(self.cols) - (self.cols - 1) / 2) * self.pitch_mm
        ys = ((self.rows - 1) / 2 - np.arange(self.rows)) * self.pitch_mm  # row 0 on top
        gx, gy = np.meshgrid(xs, ys)
        return np.stack([gx, gy], axis=-1)


_GLYPHS = {
    0: ["111", "101", "101", "101", "111"],
    1: ["010", "010", "010", "010", "010"],
    2: ["111", "001", "111", "100", "111"],
    3: ["111", "001", "111", "001", "111"],
    4: ["101", "101", "111", "001", "001"],
    5: ["111", "100", "111", "001", "111"],
    6: ["100", "100", "111", "101", "111"],
    7: ["111", "001", "001", "001", "001"],
    8: ["111", "101", "111", "101", "111"],
    9: ["111", "101", "111", "001", "001"],
}


def digit_tensor(d: int) -> np.ndarray:
    """Binary tensor (5, 3) of dtype uint8 for digit d."""
    return np.array([[int(c) for c in row] for row in _GLYPHS[d]], dtype=np.uint8)


def hamming_matrix() -> np.ndarray:
    """(10, 10) number of differing pins between every pair of digits."""
    return np.array([[int((digit_tensor(a) != digit_tensor(b)).sum()) for b in range(10)]
                     for a in range(10)])


def pin_positions_px(spec: HDSSpec, center_px, pxl_per_mm: float = 1.0) -> np.ndarray:
    """(rows, cols, 2) pin centers in hand-surface pixels."""
    return np.asarray(center_px, float) + spec.pin_offsets_mm() * pxl_per_mm


def bounding_box_px(spec: HDSSpec, center_px, pxl_per_mm: float = 1.0):
    """Outer box of the display: (xmin, ymin, xmax, ymax) in px."""
    cx, cy = center_px
    return (cx - spec.width_mm / 2 * pxl_per_mm, cy - spec.height_mm / 2 * pxl_per_mm,
            cx + spec.width_mm / 2 * pxl_per_mm, cy + spec.height_mm / 2 * pxl_per_mm)


def fit_to_region(spec: HDSSpec, region_mask_fn, search_box, pxl_per_mm=1.0, step_px=5):
    """Search for the display center that keeps the most pins inside a region.

    region_mask_fn(x, y) -> bool tells whether a hand-surface point (px) lies in the region.
    search_box = (xmin, xmax, ymin, ymax) in px. Returns (best_center, n_inside, grid), where
    grid maps every tried center to its pin count.
    """
    xmin, xmax, ymin, ymax = search_box
    best, best_n, grid = None, -1, {}
    for cx in np.arange(xmin, xmax + 1, step_px):
        for cy in np.arange(ymin, ymax + 1, step_px):
            pts = pin_positions_px(spec, (cx, cy), pxl_per_mm).reshape(-1, 2)
            n = sum(bool(region_mask_fn(x, y)) for x, y in pts)
            grid[(cx, cy)] = n
            if n > best_n:
                best, best_n = (cx, cy), n
    # several centers usually tie: take the one closest to the middle of the tied set,
    # which keeps the display away from the region edges
    tied = np.array([k for k, v in grid.items() if v == best_n], float)
    best = tuple(tied[np.argmin(((tied - tied.mean(axis=0)) ** 2).sum(axis=1))])
    return best, best_n, grid

## 2. Tests

In [ ]:
def test_digits_shape_and_binary():
    for d in range(10):
        t = digit_tensor(d)
        assert t.shape == (5, 3) and set(np.unique(t)) <= {0, 1}
    assert digit_tensor(8).sum() == 13 and digit_tensor(0).sum() == 12


def test_geometry_matches_spec():
    s = HDSSpec()
    p = pin_positions_px(s, (0, 0), 1.0)
    assert np.isclose(p[0, 1, 0] - p[0, 0, 0], 15.0)
    assert np.isclose(p[0, 0, 1] - p[1, 0, 1], 15.0)
    x0, y0, x1, y1 = bounding_box_px(s, (0, 0))
    assert (x1 - x0, y1 - y0) == (50.0, 80.0)
    assert p[..., 0].max() + 5 <= x1 and p[..., 1].max() + 5 <= y1


def test_scale_applies():
    p2 = pin_positions_px(HDSSpec(), (100, 100), 2.0)
    assert np.isclose(p2[0, 1, 0] - p2[0, 0, 0], 30.0)


def test_fit_finds_inside_position():
    inside = lambda x, y: 100 <= x <= 300 and 200 <= y <= 500
    c, n, _ = fit_to_region(HDSSpec(), inside, (80, 320, 180, 520), step_px=10)
    assert n == 15 and 100 <= c[0] <= 300


def test_timing():
    s = HDSSpec()
    assert s.ramp_s == 0.2 and s.hold_s == 0.6
    assert np.isclose(s.row_duration_s, 1.0) and np.isclose(s.duration_s, 5.0)


def test_hamming_symmetric_zero_diagonal():
    H = hamming_matrix()
    assert (H == H.T).all() and (np.diag(H) == 0).all()


for f in [test_digits_shape_and_binary, test_geometry_matches_spec, test_scale_applies,
          test_fit_finds_inside_position, test_timing, test_hamming_symmetric_zero_diagonal]:
    f()
print('HDS tests passed')

## 3. TouchSim adapter

In [ ]:
PALM_REGIONS = ["Pw1_p", "Pw2_p", "Pw3_p", "Pw4_p", "Pp1_p", "Pp2_p"]
REAL_HAND_LENGTH_MM = 189.0   # mean hand length, wrist to middle fingertip
CLASS_RATIO = {"SA1": 2, "RA": 4, "PC": 1}   # Saal et al. 2017: SA1 = 2x PC, RA = 2x SA1


def load_touchsim():
    import touchsim as ts
    return ts


def ts_px_per_mm(ts) -> float:
    """TouchSim's own scale (px per TouchSim mm, ~2.18). Never overwrite it."""
    return float(ts.hand_surface.pxl_per_mm)


def ts_hand_length_mm(ts) -> float:
    """Length of the TouchSim hand outline in TouchSim mm (~209)."""
    b = np.vstack(ts.hand_surface.boundary) / ts_px_per_mm(ts)
    return float(np.ptp(b[:, 1]))


def hand_scale(ts) -> float:
    """Uniform factor (<1) shrinking the TouchSim hand to the real length."""
    return REAL_HAND_LENGTH_MM / ts_hand_length_mm(ts)


def px_per_mm(ts) -> float:
    """Pixels per real mm on the (fixed) hand image."""
    return ts_px_per_mm(ts) / hand_scale(ts)


def pins_px(ts, spec: HDSSpec, center_px):
    """(rows, cols, 2) pin centers in pixel space."""
    return pin_positions_px(spec, center_px, px_per_mm(ts))


def pins_hand_mm(ts, spec: HDSSpec, center_px):
    """(rows, cols, 2) pin centers in TouchSim hand space (mm)."""
    p = pins_px(ts, spec, center_px)
    flat = ts.hand_surface.pixel2hand(p.reshape(-1, 2))
    return flat.reshape(p.shape)


def region_mask_fn(ts, tags=PALM_REGIONS, box=(0, 450, 0, 560)):
    """Returns f(x_px, y_px) -> bool: is the pixel inside any of the named hand regions?
    Rasterised once at 1 px so the position search is fast."""
    hs = ts.hand_surface
    x0, x1, y0, y1 = box
    xs, ys = np.arange(x0, x1 + 1), np.arange(y0, y1 + 1)
    gx, gy = np.meshgrid(xs, ys, indexing="ij")
    pts_px = np.column_stack([gx.ravel(), gy.ravel()]).astype(float)
    _, ids = hs.locate(hs.pixel2hand(pts_px))
    want = [hs.tags.index(t) for t in tags]
    inside = np.isin(ids, want).reshape(gx.shape)

    def f(x, y):
        i, j = int(round(x)) - x0, int(round(y)) - y0
        return 0 <= i < inside.shape[0] and 0 <= j < inside.shape[1] and bool(inside[i, j])
    return f


def build_stimulus(ts, tensor, spec: HDSSpec, center_px, fs=5000):
    """Row-by-row stimulus. Row 0 (top) first, then the next rows; within a row all active pins
    move together: ramp down (ramp_s), hold (hold_s), ramp up (ramp_s), then the next row starts."""
    hand = pins_hand_mm(ts, spec, center_px)
    pin_radius_ts = spec.pin_diameter_mm / 2 / hand_scale(ts)   # real mm -> TouchSim mm
    n_ramp, n_hold = int(round(spec.ramp_s * fs)), int(round(spec.hold_s * fs))
    pulse = spec.depth_mm * np.concatenate([np.linspace(0, 1, n_ramp), np.ones(n_hold),
                                            np.linspace(1, 0, n_ramp)])
    row_n = len(pulse)
    total_n = row_n * tensor.shape[0]
    locs, traces = [], []
    for r in range(tensor.shape[0]):
        for c in range(tensor.shape[1]):
            if not tensor[r, c]:
                continue
            trace = np.zeros(total_n)
            trace[r * row_n:(r + 1) * row_n] = pulse
            locs.append(hand[r, c])
            traces.append(trace)
    if not traces:
        raise ValueError("Tensor has no active pins.")
    return ts.Stimulus(location=np.array(locs), trace=np.array(traces), fs=fs,
                       pin_radius=pin_radius_ts)


def balance_classes(pop, ratio=CLASS_RATIO, seed=0):
    """Randomly subsample afferents so the class counts follow `ratio` (as many as possible)."""
    cls = np.array(pop.affclass)
    unit = min((cls == k).sum() / w for k, w in ratio.items())
    rng = np.random.default_rng(seed)
    sel = np.zeros(len(cls), bool)
    for k, w in ratio.items():
        sel[rng.choice(np.where(cls == k)[0], int(round(unit * w)), replace=False)] = True
    return pop[sel]


def make_population(ts, spec: HDSSpec, center_px, margin_mm=20.0, density_multiplier=1.0,
                    class_ratio=CLASS_RATIO, seed=0):
    """Afferents (SA1, RA, PC) under / near the HDS box (+ margin), class-balanced.
    Build once per HDS position and reuse for every digit so responses are comparable."""
    pop = ts.affpop_hand(density_multiplier=density_multiplier)
    ppm = px_per_mm(ts)
    loc_px = ts.hand_surface.hand2pixel(pop.location)
    x0, y0, x1, y1 = bounding_box_px(spec, center_px, ppm)
    m = margin_mm * ppm
    keep = ((loc_px[:, 0] >= x0 - m) & (loc_px[:, 0] <= x1 + m) &
            (loc_px[:, 1] >= y0 - m) & (loc_px[:, 1] <= y1 + m))
    if not keep.any():
        raise ValueError("No afferents near the HDS - is the center on the hand?")
    pop = pop[keep]
    return balance_classes(pop, class_ratio, seed) if class_ratio else pop


def respond(ts, stim, pop):
    return pop.response(stim)

## 4. Evaluation and plotting helpers

In [ ]:
def spike_counts(resp) -> np.ndarray:
    """Number of spikes of every afferent."""
    return np.array([len(s) for s in resp.spikes])


def spike_counts_by_row(resp, n_rows, row_duration_s) -> np.ndarray:
    """(n_rows, N) spikes of every afferent inside the time window of each HDS row."""
    edges = np.arange(n_rows + 1) * row_duration_s
    return np.array([[np.count_nonzero((s >= edges[r]) & (s < edges[r + 1])) for s in resp.spikes]
                     for r in range(n_rows)])


def evaluate(counts_a, counts_b) -> float:
    """Pearson r between two spike-count vectors (0.0 if either is constant)."""
    if np.std(counts_a) == 0 or np.std(counts_b) == 0:
        return 0.0
    return float(np.corrcoef(counts_a, counts_b)[0, 1])


def similarity_matrix(all_counts: dict) -> np.ndarray:
    """(n, n) Pearson r between the responses of every pair of digits."""
    keys = sorted(all_counts)
    return np.array([[evaluate(all_counts[a], all_counts[b]) for b in keys] for a in keys])


def counts_by_class(counts, pop) -> dict:
    """Total spikes per afferent class."""
    cls = np.array(pop.affclass)
    return {k: int(counts[cls == k].sum()) for k in ("SA1", "RA", "PC")}


def plot_layout(ts, center_px, digit=3, spec=None, highlight=PALM_REGIONS, ax=None):
    """Hand (palm regions shaded) and the HDS pattern of `digit`, axes in real mm."""
    spec = spec or HDSSpec()
    hs, ppm = ts.hand_surface, px_per_mm(ts)
    ax = ax or plt.subplots(figsize=(5, 6))[1]
    for b, poly in enumerate(hs.boundary):
        ax.add_patch(Polygon(poly / ppm, closed=True, ec="k", lw=1,
                             fc="#fde9c9" if hs.tags[b] in highlight else "none"))
    x0, y0, x1, y1 = np.array(bounding_box_px(spec, center_px, ppm)) / ppm
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="tab:blue", lw=1.5))
    t, pos = digit_tensor(digit), pin_positions_px(spec, center_px, ppm) / ppm
    for r in range(spec.rows):
        for c in range(spec.cols):
            ax.add_patch(Circle(pos[r, c], spec.pin_diameter_mm / 2,
                                fc="tab:red" if t[r, c] else "none", ec="k", alpha=.8, lw=.8))
    allb = np.vstack(hs.boundary) / ppm
    ax.set_xlim(allb[:, 0].min() - 5, allb[:, 0].max() + 5)
    ax.set_ylim(allb[:, 1].min() - 5, allb[:, 1].max() + 5)
    ax.set_aspect("equal"); ax.set_xlabel("mm"); ax.set_title(f"HDS digit {digit}")
    return ax

## 5. Run: place HDS, build population

In [ ]:
spec = HDSSpec()
PPM = px_per_mm(ts)
print(f"TouchSim hand: {ts_hand_length_mm(ts):.1f} mm, {ts_px_per_mm(ts):.3f} px/mm")
print(f"HAND_SCALE ({REAL_HAND_LENGTH_MM:.0f} mm) = {hand_scale(ts):.4f} -> {PPM:.3f} px per real mm")
print(f"HDS {spec.width_mm:.0f}x{spec.height_mm:.0f} mm = {spec.width_mm*PPM:.0f}x{spec.height_mm*PPM:.0f} px")
print(f"Timing per row: down {spec.ramp_s}s + hold {spec.hold_s}s + up {spec.ramp_s}s = {spec.row_duration_s}s; "
      f"{spec.rows} rows = {spec.duration_s}s")

best, n, grid = fit_to_region(spec, region_mask_fn(ts), (40, 400, 100, 520), pxl_per_mm=PPM, step_px=4)
CENTER = (float(best[0]), float(best[1]))
print(f"HDS center (px): {CENTER} -> {n}/15 pins inside the 6 palm regions")

pop = make_population(ts, spec, CENTER, density_multiplier=1.0)
print(pop)
plot_layout(ts, CENTER, digit=3, spec=spec); plt.show()

## 6. Simulate digits 0–9 (row by row)

In [ ]:
all_counts, all_resp = {}, {}
for d in range(10):
    tensor = digit_tensor(d)
    resp = respond(ts, build_stimulus(ts, tensor, spec, CENTER), pop)
    all_resp[d], all_counts[d] = resp, spike_counts(resp)
    by_row = spike_counts_by_row(resp, spec.rows, spec.row_duration_s).sum(axis=1)
    print(f"digit {d}: {int(tensor.sum()):2d} pins, {all_counts[d].sum():6d} spikes, "
          f"per row {by_row.tolist()}, {counts_by_class(all_counts[d], pop)}")

## 7. Similarity between digit responses

In [ ]:
S = similarity_matrix(all_counts)
H = hamming_matrix()
np.set_printoptions(precision=2, linewidth=140)
print("Pearson r between digits:\n", S)
iu = np.triu_indices(10, 1)
print(f"corr(pin difference, response similarity) = {np.corrcoef(H[iu], S[iu])[0, 1]:.2f}")